In [14]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated 
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

In [15]:
llm = ChatOllama(
    model="llama3",
    temperature=0,
    streaming=False
)

In [16]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explaination : str

In [17]:
def generate_joke(state : JokeState):
    prompt = f"Generate a joke on the {state['topic']}"
    response = llm.invoke(prompt).content
    return {'joke':response}

In [18]:
def generate_explaination(state : JokeState):
    prompt = f"Generate an explaination for the {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explaination':response}

In [20]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [23]:
config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({"topic": "pizza"}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty!',
 'explaination': 'The classic joke!\n\nThe punchline "Because it was feeling a little crusty!" is a play on words, using a pun to create humor. Here\'s a breakdown of why it\'s funny:\n\n* "Why did the pizza go to therapy?" sets up the expectation that the answer will be a serious reason, like a human would go to therapy (e.g., emotional trauma, relationship issues).\n* "Because it was feeling a little crusty!" subverts that expectation by using a wordplay. "Crusty" has a double meaning:\n\t+ In the context of pizza, "crusty" refers to the outer layer of the crust, which is a normal and desirable characteristic of a pizza.\n\t+ In the context of human emotions, "crusty" means irritable, grumpy, or having a bad attitude (e.g., "I\'m feeling a little crusty today").\n\nThe humor comes from the unexpected twist on the word "crusty." The joke relies on the listener being familiar with

In [24]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty!', 'explaination': 'The classic joke!\n\nThe punchline "Because it was feeling a little crusty!" is a play on words, using a pun to create humor. Here\'s a breakdown of why it\'s funny:\n\n* "Why did the pizza go to therapy?" sets up the expectation that the answer will be a serious reason, like a human would go to therapy (e.g., emotional trauma, relationship issues).\n* "Because it was feeling a little crusty!" subverts that expectation by using a wordplay. "Crusty" has a double meaning:\n\t+ In the context of pizza, "crusty" refers to the outer layer of the crust, which is a normal and desirable characteristic of a pizza.\n\t+ In the context of human emotions, "crusty" means irritable, grumpy, or having a bad attitude (e.g., "I\'m feeling a little crusty today").\n\nThe humor comes from the unexpected twist on the word "crusty." The joke relies on the listener 

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to therapy?\n\nBecause it was feeling a little crusty!', 'explaination': 'The classic joke!\n\nThe punchline "Because it was feeling a little crusty!" is a play on words, using a pun to create humor. Here\'s a breakdown of why it\'s funny:\n\n* "Why did the pizza go to therapy?" sets up the expectation that the answer will be a serious reason, like a human would go to therapy (e.g., emotional trauma, relationship issues).\n* "Because it was feeling a little crusty!" subverts that expectation by using a wordplay. "Crusty" has a double meaning:\n\t+ In the context of pizza, "crusty" refers to the outer layer of the crust, which is a normal and desirable characteristic of a pizza.\n\t+ In the context of human emotions, "crusty" means irritable, grumpy, or having a bad attitude (e.g., "I\'m feeling a little crusty today").\n\nThe humor comes from the unexpected twist on the word "crusty." The joke relies on the listener